# BÀI TẬP LỚN: KHÁM PHÁ VÀ SO SÁNH CÁC THUẬT TOÁN TỐI ƯU HÓA (OPTIMIZERS)
## Gradient Descent, Momentum, AdaGrad, RMSProp và Adam

---
### 🎯 Mục tiêu bài tập:
1. **Hiểu bản chất toán học** và cơ chế cập nhật trọng số của các thuật toán tối ưu hóa nền tảng trong Deep Learning.
2. **Tự lập trình từ đầu (From Scratch)** hoàn toàn bằng thư viện **NumPy** mà không dùng các framework tự động vi phân.
3. **Thử thách trên Loss Landscape phức tạp cao cấp (Multi-Basin Rugged Landscape)** với nhiều dạng địa hình hiểm trở:
   - **Đỉnh đồi dốc đứng (Steep Peak)** bên phải gây bùng nổ gradient nếu bước quá lớn.
   - **Vùng yên ngựa kéo dài và hẹp (Narrow Curved Ravine / Valley)** mô phỏng hẻm núi dạng Rosenbrock khiến các thuật toán không thích nghi dễ bị dao động hỗn loạn (oscillation).
   - **Nhiều bẫy cực tiểu địa phương cạn (Local Minima Traps)** và gợn sóng lượn sóng (High-frequency Ripples) gây bẫy gradient.
   - **Vùng cao nguyên phẳng (Flat Saddle Plateau)** với gradient triệt tiêu $\sigma^2 \approx 0.001$.
4. **Kiểm thử trên ĐA DẠNG ĐIỂM KHỞI TẠO (Multiple Starting Points)**:
   - Điểm 1: **Vùng yên ngựa phẳng** $(0.1, -0.1)$ - Thử thách gradient phẳng / vanishing gradient.
   - Điểm 2: **Sườn dốc hiểm trở gần đỉnh núi** $(2.5, 1.5)$ - Thử thách vọt đà, vượt qua sườn đồi dốc cao.
   - Điểm 3: **Vùng hẻm núi cong & mấp mô** $(-3.5, 1.0)$ - Thử thách thoát khỏi bẫy hố cạn và dập tắt dao động ngoằn ngoèo.
   - Điểm 4: **Góc đối diện xa** $(-1.5, -4.5)$ - Thử thách tìm đường đến cực tiểu toàn cục qua bề mặt uốn lượn.
5. **Trực quan hóa đường đi (Trajectory)** trên không gian 3D Surface và 2D Contour Plot qua từng kịch bản xuất phát.
6. **Phân tích định lượng & so sánh hiệu năng toàn diện**.

## PHẦN 1: KHỞI TẠO KHÔNG GIAN HÀM LOSS PHỨC TẠP (COMPLEX LOSS LANDSCAPE)

Không gian hàm mục tiêu $L(w_1, w_2)$ được thiết kế nâng cao bao gồm 4 thành phần địa hình:
1. **Khe núi cong hẹp (Ravine / Valley)**: Dạng hẻm núi kéo dài dọc $w_2 \approx 0.15 w_1^2 + 0.3 w_1$ tạo độ cong dị hướng mạnh (anisotropic curvature) – nguyên nhân gây dao động dội qua lại hai bên sườn núi.
2. **Đỉnh núi dốc đứng (Steep Peak)**: Nằm tại $(3.3, 0.0)$ với độ cao loss $\sim 22 - 24$.
3. **4 Hố cực tiểu với độ sâu khác nhau**:
   - 3 cực tiểu địa phương (Local Minima - bẫy cạn) tại $(-2.5, 2.5)$, $(0.25, 2.3)$ và $(-2.5, -3.5)$.
   - 1 cực tiểu sâu nhất (Cực tiểu toàn cục - Global Minimum) tại $(0.5, -3.5)$ với giá trị loss tối ưu.
4. **Vùng yên ngựa phẳng (Saddle Point)**: Tại $(0.1, -0.1)$ với độ dốc siêu nhỏ $(\sigma^2 \approx 0.001)$ kèm các gợn sóng phụ tần số cao.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Thiet lap seed de dam bao tinh tai lap
np.random.seed(42)

def loss_function(w1, w2):
    base = 0.05 * w1**2 + 0.04 * (w2 + 1.0)**2
    peak = 18.0 * np.exp(-((w1 - 3.3)**2 / 1.5 + (w2 - 0.0)**2 / 1.8))
    b_deep = -2.5 * np.exp(-((w1 + 2.5)**2 / 2.0 + (w2 - 2.5)**2 / 2.0))
    b_shallow = -1.2 * np.exp(-((w1 - 0.2)**2 / 1.0 + (w2 - 2.0)**2 / 1.0))
    b_puddle = -0.8 * np.exp(-((w1 - 0.1)**2 / 0.4 + (w2 + 1.0)**2 / 0.4))
    b_global = -4.5 * np.exp(-((w1 - 0.5)**2 / 2.0 + (w2 + 3.5)**2 / 2.0))
    return base + peak + b_deep + b_shallow + b_puddle + b_global

def compute_gradients(w1, w2):
    db_dw1 = 0.10 * w1
    db_dw2 = 0.08 * (w2 + 1.0)
    
    exp_peak = np.exp(-((w1 - 3.3)**2 / 1.5 + (w2 - 0.0)**2 / 1.8))
    dp_dw1 = 18.0 * exp_peak * (-2.0 * (w1 - 3.3) / 1.5)
    dp_dw2 = 18.0 * exp_peak * (-2.0 * (w2 - 0.0) / 1.8)
    
    exp_deep = np.exp(-((w1 + 2.5)**2 / 2.0 + (w2 - 2.5)**2 / 2.0))
    dd_dw1 = -2.5 * exp_deep * (-2.0 * (w1 + 2.5) / 2.0)
    dd_dw2 = -2.5 * exp_deep * (-2.0 * (w2 - 2.5) / 2.0)
    
    exp_sh = np.exp(-((w1 - 0.2)**2 / 1.0 + (w2 - 2.0)**2 / 1.0))
    ds_dw1 = -1.2 * exp_sh * (-2.0 * (w1 - 0.2) / 1.0)
    ds_dw2 = -1.2 * exp_sh * (-2.0 * (w2 - 2.0) / 1.0)
    
    exp_pud = np.exp(-((w1 - 0.1)**2 / 0.4 + (w2 + 1.0)**2 / 0.4))
    dpu_dw1 = -0.8 * exp_pud * (-2.0 * (w1 - 0.1) / 0.4)
    dpu_dw2 = -0.8 * exp_pud * (-2.0 * (w2 + 1.0) / 0.4)
    
    exp_glob = np.exp(-((w1 - 0.5)**2 / 2.0 + (w2 + 3.5)**2 / 2.0))
    dg_dw1 = -4.5 * exp_glob * (-2.0 * (w1 - 0.5) / 2.0)
    dg_dw2 = -4.5 * exp_glob * (-2.0 * (w2 + 3.5) / 2.0)
    
    dw1 = db_dw1 + dp_dw1 + dd_dw1 + ds_dw1 + dpu_dw1 + dg_dw1
    dw2 = db_dw2 + dp_dw2 + dd_dw2 + ds_dw2 + dpu_dw2 + dg_dw2
    return np.array([dw1, dw2], dtype=float)

start_scenarios = {
    "Vùng yên ngựa phẳng (Saddle Flat)": np.array([0.1, -0.1]),
    "Sườn dốc gần đỉnh núi (Steep Slope)": np.array([2.5, 1.5]),
    "Hẻm núi mấp mô (Rugged Ravine)": np.array([-2.5, 2.0]),
    "Góc đối diện xa (Far Basin Area)": np.array([-1.5, -4.0])
}



print("Khởi tạo không gian hàm Loss phân hóa thành công!")


### Trực quan hóa bản đồ địa hình & Các điểm xuất phát thử nghiệm

In [ ]:
w1_vals = np.linspace(-5, 5, 250)
w2_vals = np.linspace(-5, 5, 250)
W1, W2 = np.meshgrid(w1_vals, w2_vals)
Z = loss_function(W1, W2)

fig = plt.figure(figsize=(18, 8))

# 1. Biểu đồ 3D Surface View: Địa hình Loss & Các điểm khởi tạo
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
surf = ax1.plot_surface(W1, W2, Z, cmap="viridis", edgecolor="none", alpha=0.8)

start_colors = ["lime", "cyan", "magenta", "orange"]
for (label, pt), col in zip(start_scenarios.items(), start_colors):
    z_pt = loss_function(pt[0], pt[1])
    ax1.scatter(pt[0], pt[1], z_pt, color=col, edgecolors="black", s=110, marker="D", label=f"Start: {label}", zorder=10)

# Đánh dấu Global Min trên 3D
z_global = loss_function(0.5, -3.5)
ax1.scatter([0.5], [-3.5], [z_global], color="red", edgecolors="white", s=160, marker="*", label="Global Min (0.5, -3.5)", zorder=11)

ax1.set_title("Không Gian Loss Landscape 3D & Các Điểm Khởi Tạo", fontsize=13, fontweight="bold")
ax1.set_xlabel("$w_1$", fontsize=11)
ax1.set_ylabel("$w_2$", fontsize=11)
ax1.set_zlabel("Loss", fontsize=11)
ax1.view_init(elev=50, azim=-118)
ax1.legend(loc="upper right", fontsize=8.5, framealpha=0.9)

# 2. Biểu đồ 2D Contour View
ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contourf(W1, W2, Z, levels=35, cmap="viridis")
lines = ax2.contour(W1, W2, Z, levels=14, colors="black", alpha=0.35, linewidths=0.7)
ax2.clabel(lines, inline=True, fontsize=8)
fig.colorbar(contour, ax=ax2, label="Loss", fraction=0.046, pad=0.04)

for (label, pt), col in zip(start_scenarios.items(), start_colors):
    ax2.scatter(pt[0], pt[1], color=col, edgecolors="black", s=100, marker="D", label=f"Start: {label}", zorder=5)

ax2.scatter([0.5], [-3.5], color="red", edgecolors="white", s=150, marker="*", label="Global Min (0.5, -3.5)", zorder=6)

ax2.set_title("Bản Đồ 2D Contour & Các Điểm Khởi Tạo", fontsize=13, fontweight="bold")
ax2.set_xlabel("$w_1$", fontsize=11)
ax2.set_ylabel("$w_2$", fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
from abc import ABC, abstractmethod
from typing import Callable, Tuple
import numpy as np

class BaseOptimizer(ABC):
    def __init__(self, loss_fn: Callable, gradient_fn: Callable):
        self.loss_fn = loss_fn
        self.gradient_fn = gradient_fn 
        
    @abstractmethod    
    def run(self, w_start: np.ndarray, num_epochs=140) -> Tuple[np.ndarray, np.ndarray]:
        """
        Chạy thuật toán tối ưu hóa từ điểm khởi tạo.
        Args:
            w_start (np.ndarray): Tọa độ điểm bắt đầu, ví dụ np.array([w1, w2]).
            num_epochs (int, optional): Số vòng lặp tối ưu. Mặc định là 140.
        Returns:
            Tuple[np.ndarray, np.ndarray]:
                - history_w (np.ndarray, shape=(num_epochs + 1, len(w_start))):
                    Mảng chứa tọa độ trọng số w tại điểm xuất phát và sau mỗi epoch (Trajectory).
                - history_loss (np.ndarray, shape=(num_epochs + 1,)):
                    Mảng chứa giá trị hàm Loss tương ứng tại mỗi bước (Loss curve).
        """
        pass 

# Bí danh tương thích ngược
BaseOptmizer = BaseOptimizer


---
## PHẦN 2: TỰ LẬP TRÌNH CÁC BỘ TỐI ƯU HÓA (FROM SCRATCH BẰNG NUMPY)

Sinh viên hãy hoàn thiện các class Optimizer bên dưới chỉ bằng thư viện **NumPy**.

### 1. Gradient Descent (Vanilla GD)
Quy tắc cập nhật:
$$w_{t+1} = w_t - \eta \nabla L(w_t)$$
với $\eta$ là tốc độ học (learning rate).

In [ ]:
class GradientDescent(BaseOptimizer):
    def __init__(self, loss_fn: Callable, gradient_fn: Callable, lr=0.04):
        super().__init__(loss_fn, gradient_fn)
        self.lr = lr 
        
    def run(self, w_start: np.ndarray, num_epochs=140, verbose=False):
        w = np.copy(w_start).astype(float)
        history_w = [np.copy(w_start)]
        history_loss = [self.loss_fn(w_start[0], w_start[1])]
        for epoch in range(num_epochs):
            grad = self.gradient_fn(w[0], w[1])
            grad = np.clip(grad, -100.0, 100.0)
            
            w = w - self.lr * grad
            new_loss = self.loss_fn(w[0], w[1])
            if verbose and epoch % 20 == 0:
                print(f"Loss = {new_loss}")
            history_w.append(np.copy(w))
            history_loss.append(new_loss)
        return np.array(history_w), np.array(history_loss)


#### 📊 Trực quan hóa hoạt động của Gradient Descent qua 4 kịch bản xuất phát


In [ ]:
# Trực quan hóa độc lập cho Gradient Descent trên 2D Contour và Đường cong Loss
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. 2D Contour View
ax_contour = axes[0]
ax_contour.contourf(W1, W2, Z, levels=35, cmap="viridis", alpha=0.9)
c_lines = ax_contour.contour(W1, W2, Z, levels=14, colors="black", alpha=0.3, linewidths=0.6)
ax_contour.clabel(c_lines, inline=True, fontsize=8)

# 2. Loss curve
ax_loss = axes[1]

sc_colors = {
    "Vùng yên ngựa phẳng (Saddle Flat)": "#4ade80",
    "Sườn dốc gần đỉnh núi (Steep Slope)": "#38bdf8",
    "Hẻm núi mấp mô (Rugged Ravine)": "#e879f9",
    "Góc đối diện xa (Far Basin Area)": "#fbbf24"
}

for sc_name, pt in start_scenarios.items():
    opt_instance = GradientDescent(
        lr=0.04,
        loss_fn=loss_function,
        gradient_fn=compute_gradients
    )
    traj, losses = opt_instance.run(pt, num_epochs=140)
    col = sc_colors[sc_name]
    
    # Vẽ trên Contour
    ax_contour.plot(traj[:, 0], traj[:, 1], color=col, linewidth=2.2, label=f"Path: {sc_name}")
    ax_contour.scatter(traj[0, 0], traj[0, 1], color=col, edgecolors="black", s=90, marker="D", zorder=5)
    ax_contour.scatter(traj[-1, 0], traj[-1, 1], color=col, edgecolors="red", s=110, marker="o", zorder=6)
    
    # Vẽ trên Loss curve
    ax_loss.plot(losses, color=col, linewidth=2.0, label=f"Loss: {sc_name}")

ax_contour.scatter([0.5], [-3.5], color="red", edgecolors="white", s=160, marker="*", label="Global Min (0.5, -3.5)", zorder=7)
ax_contour.set_title("Gradient Descent - 2D Contour Trajectory View", fontsize=13, fontweight="bold")
ax_contour.set_xlabel("$", fontsize=11)
ax_contour.set_ylabel("$", fontsize=11)
ax_contour.set_xlim(-5, 5)
ax_contour.set_ylim(-5, 5)
ax_contour.legend(loc="lower left", fontsize=8.5, framealpha=0.9)

ax_loss.set_title("Gradient Descent - Loss Decay Curve Across 4 Scenarios", fontsize=13, fontweight="bold")
ax_loss.set_xlabel("Epoch", fontsize=11)
ax_loss.set_ylabel("Loss Value", fontsize=11)
ax_loss.grid(True, linestyle="--", alpha=0.5)
ax_loss.legend(fontsize=8.5, framealpha=0.9)

plt.tight_layout()
plt.show()


### 2. Momentum Optimizer
Momentum bổ sung "quán tính" vào vector vận tốc $v$, giúp vượt qua bề mặt bằng phẳng hoặc thoát hố cạn và giảm dao động:
$$v_{t+1} = \gamma v_t + \eta \nabla L(w_t)$$
$$w_{t+1} = w_t - v_{t+1}$$
với $\gamma \in [0, 1)$ là hệ số momentum (thường chọn $0.9$).

In [ ]:
class Momentum(BaseOptimizer):
    def __init__(
        self,
        loss_fn: Callable,
        gradient_fn: Callable,
        v_0: float | None = None,
        lr=0.035,
        gamma=0.88,
    ):
        super().__init__(loss_fn, gradient_fn)
        self.lr = lr
        self.gamma = gamma
        self.v = v_0

    def run(self, w_start, num_epochs=140):
        """
        Cập nhật trọng số w với quán tính Momentum.
        """
        if self.v is None:
            self.v = np.zeros_like(w_start)

        # TODO: YOUR CODE HERE
        pass


#### 📊 Trực quan hóa hoạt động của Momentum qua 4 kịch bản xuất phát


In [ ]:
# Trực quan hóa độc lập cho Momentum trên 2D Contour và Đường cong Loss
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. 2D Contour View
ax_contour = axes[0]
ax_contour.contourf(W1, W2, Z, levels=35, cmap="viridis", alpha=0.9)
c_lines = ax_contour.contour(W1, W2, Z, levels=14, colors="black", alpha=0.3, linewidths=0.6)
ax_contour.clabel(c_lines, inline=True, fontsize=8)

# 2. Loss curve
ax_loss = axes[1]

sc_colors = {
    "Vùng yên ngựa phẳng (Saddle Flat)": "#4ade80",
    "Sườn dốc gần đỉnh núi (Steep Slope)": "#38bdf8",
    "Hẻm núi mấp mô (Rugged Ravine)": "#e879f9",
    "Góc đối diện xa (Far Basin Area)": "#fbbf24"
}

for sc_name, pt in start_scenarios.items():
    opt_instance = Momentum(loss_fn=loss_function, 
                            gradient_fn=compute_gradients,
                            lr=0.035, 
                            gamma=0.88)
    traj, losses = opt_instance.run( pt, num_epochs=140)
    col = sc_colors[sc_name]
    
    # Vẽ trên Contour
    ax_contour.plot(traj[:, 0], traj[:, 1], color=col, linewidth=2.2, label=f"Path: {sc_name}")
    ax_contour.scatter(traj[0, 0], traj[0, 1], color=col, edgecolors="black", s=90, marker="D", zorder=5)
    ax_contour.scatter(traj[-1, 0], traj[-1, 1], color=col, edgecolors="red", s=110, marker="^", zorder=6)
    
    # Vẽ trên Loss curve
    ax_loss.plot(losses, color=col, linewidth=2.0, label=f"Loss: {sc_name}")

ax_contour.scatter([0.5], [-3.5], color="red", edgecolors="white", s=160, marker="*", label="Global Min (0.5, -3.5)", zorder=7)
ax_contour.set_title("Momentum - 2D Contour Trajectory View", fontsize=13, fontweight="bold")
ax_contour.set_xlabel("$w_1$", fontsize=11)
ax_contour.set_ylabel("$w_2$", fontsize=11)
ax_contour.set_xlim(-5, 5)
ax_contour.set_ylim(-5, 5)

ax_loss.set_title("Momentum - Loss Decay Curve Across 4 Scenarios", fontsize=13, fontweight="bold")
ax_loss.set_xlabel("Epoch", fontsize=11)
ax_loss.set_ylabel("Loss Value", fontsize=11)
ax_loss.grid(True, linestyle="--", alpha=0.5)
ax_loss.legend(fontsize=8.5, framealpha=0.9)

plt.tight_layout()
plt.show()


### 3. AdaGrad (Adaptive Gradient Algorithm)
AdaGrad tự động điều chỉnh tốc độ học riêng cho từng tham số dựa trên tổng bình phương các gradient trong lịch sử:
$$G_{t+1} = G_t + (\nabla L(w_t))^2$$
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{G_{t+1} + \epsilon}} \odot \nabla L(w_t)$$
với $\epsilon$ là hằng số dương nhỏ (vd: $10^{-8}$) tránh chia cho 0, $\odot$ là phép nhân từng phần tử.

In [ ]:
class AdaGrad(BaseOptimizer):
    def __init__(
        self, loss_fn: Callable, gradient_fn: Callable, lr=0.25, eps=1e-8
    ):
        super().__init__(loss_fn, gradient_fn)
        self.lr = lr
        self.eps = eps
        self.G = None

    def run(self, w_start, num_epochs=140):
        """
        Cập nhật trọng số w theo thuật toán AdaGrad.
        """
        if self.G is None:
            self.G = np.zeros_like(w_start)

        # TODO: YOUR CODE HERE
        pass


#### 📊 Trực quan hóa hoạt động của AdaGrad qua 4 kịch bản xuất phát


In [ ]:
# Trực quan hóa độc lập cho AdaGrad trên 2D Contour và Đường cong Loss
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. 2D Contour View
ax_contour = axes[0]
ax_contour.contourf(W1, W2, Z, levels=35, cmap="viridis", alpha=0.9)
c_lines = ax_contour.contour(W1, W2, Z, levels=14, colors="black", alpha=0.3, linewidths=0.6)
ax_contour.clabel(c_lines, inline=True, fontsize=8)

# 2. Loss curve
ax_loss = axes[1]

sc_colors = {
    "Vùng yên ngựa phẳng (Saddle Flat)": "#4ade80",
    "Sườn dốc gần đỉnh núi (Steep Slope)": "#38bdf8",
    "Hẻm núi mấp mô (Rugged Ravine)": "#e879f9",
    "Góc đối diện xa (Far Basin Area)": "#fbbf24"
}

for sc_name, pt in start_scenarios.items():
    opt_instance = AdaGrad(loss_fn=loss_function, gradient_fn=compute_gradients, lr=0.25)
    traj, losses = opt_instance.run(pt, num_epochs=140)
    col = sc_colors[sc_name]
    
    # Vẽ trên Contour
    ax_contour.plot(traj[:, 0], traj[:, 1], color=col, linewidth=2.2, label=f"Path: {sc_name}")
    ax_contour.scatter(traj[0, 0], traj[0, 1], color=col, edgecolors="black", s=90, marker="D", zorder=5)
    ax_contour.scatter(traj[-1, 0], traj[-1, 1], color=col, edgecolors="red", s=110, marker="v", zorder=6)
    
    # Vẽ trên Loss curve
    ax_loss.plot(losses, color=col, linewidth=2.0, label=f"Loss: {sc_name}")

ax_contour.scatter([0.5], [-3.5], color="red", edgecolors="white", s=160, marker="*", label="Global Min (0.5, -3.5)", zorder=7)
ax_contour.set_title("AdaGrad - 2D Contour Trajectory View", fontsize=13, fontweight="bold")
ax_contour.set_xlabel("$w_1$", fontsize=11)
ax_contour.set_ylabel("$w_2$", fontsize=11)
ax_contour.set_xlim(-5, 5)
ax_contour.set_ylim(-5, 5)
ax_contour.legend(loc="lower left", fontsize=8.5, framealpha=0.9)

ax_loss.set_title("AdaGrad - Loss Decay Curve Across 4 Scenarios", fontsize=13, fontweight="bold")
ax_loss.set_xlabel("Epoch", fontsize=11)
ax_loss.set_ylabel("Loss Value", fontsize=11)
ax_loss.grid(True, linestyle="--", alpha=0.5)
ax_loss.legend(fontsize=8.5, framealpha=0.9)

plt.tight_layout()
plt.show()


### 4. RMSProp (Root Mean Square Propagation)
RMSProp khắc phục hiện tượng triệt tiêu learning rate của AdaGrad bằng cách lấy trung bình trượt có trọng số mũ:
$$v_{t+1} = \beta v_t + (1 - \beta) (\nabla L(w_t))^2$$
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{v_{t+1} + \epsilon}} \odot \nabla L(w_t)$$
với hệ số suy giảm $\beta$ (thường là $0.9$).

In [ ]:
class RMSProp(BaseOptimizer):
    def __init__(
        self, loss_fn: Callable, gradient_fn: Callable, lr=0.035, beta=0.9, eps=1e-8
    ):
        super().__init__(loss_fn, gradient_fn)
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.v = None

    def run(self, w_start, num_epochs=140):
        """
        Cập nhật trọng số w theo thuật toán RMSProp.
        """
        if self.v is None:
            self.v = np.zeros_like(w_start)

        # TODO: YOUR CODE HERE
        pass


#### 📊 Trực quan hóa hoạt động của RMSProp qua 4 kịch bản xuất phát


In [ ]:
# Trực quan hóa độc lập cho RMSProp trên 2D Contour và Đường cong Loss
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. 2D Contour View
ax_contour = axes[0]
ax_contour.contourf(W1, W2, Z, levels=35, cmap="viridis", alpha=0.9)
c_lines = ax_contour.contour(W1, W2, Z, levels=14, colors="black", alpha=0.3, linewidths=0.6)
ax_contour.clabel(c_lines, inline=True, fontsize=8)

# 2. Loss curve
ax_loss = axes[1]

sc_colors = {
    "Vùng yên ngựa phẳng (Saddle Flat)": "#4ade80",
    "Sườn dốc gần đỉnh núi (Steep Slope)": "#38bdf8",
    "Hẻm núi mấp mô (Rugged Ravine)": "#e879f9",
    "Góc đối diện xa (Far Basin Area)": "#fbbf24"
}

for sc_name, pt in start_scenarios.items():
    opt_instance = RMSProp(loss_fn=loss_function, gradient_fn=compute_gradients, lr=0.035, beta=0.9)
    traj, losses = opt_instance.run( pt, num_epochs=140)
    col = sc_colors[sc_name]
    
    # Vẽ trên Contour
    ax_contour.plot(traj[:, 0], traj[:, 1], color=col, linewidth=2.2, label=f"Path: {sc_name}")
    ax_contour.scatter(traj[0, 0], traj[0, 1], color=col, edgecolors="black", s=90, marker="D", zorder=5)
    ax_contour.scatter(traj[-1, 0], traj[-1, 1], color=col, edgecolors="red", s=110, marker="x", zorder=6)
    
    # Vẽ trên Loss curve
    ax_loss.plot(losses, color=col, linewidth=2.0, label=f"Loss: {sc_name}")

ax_contour.scatter([0.5], [-3.5], color="red", edgecolors="white", s=160, marker="*", label="Global Min (0.5, -3.5)", zorder=7)
ax_contour.set_title("RMSProp - 2D Contour Trajectory View", fontsize=13, fontweight="bold")
ax_contour.set_xlabel("$w_1$", fontsize=11)
ax_contour.set_ylabel("$w_2$", fontsize=11)
ax_contour.set_xlim(-5, 5)
ax_contour.set_ylim(-5, 5)
ax_contour.legend(loc="lower left", fontsize=8.5, framealpha=0.9)

ax_loss.set_title("RMSProp - Loss Decay Curve Across 4 Scenarios", fontsize=13, fontweight="bold")
ax_loss.set_xlabel("Epoch", fontsize=11)
ax_loss.set_ylabel("Loss Value", fontsize=11)
ax_loss.grid(True, linestyle="--", alpha=0.5)
ax_loss.legend(fontsize=8.5, framealpha=0.9)

plt.tight_layout()
plt.show()


### 5. Adam (Adaptive Moment Estimation)
Adam kết hợp cả **Momentum** (mô men bậc 1 $m_t$) và **RMSProp** (mô men bậc 2 $v_t$), kèm hiệu chỉnh độ chệch (bias correction):
$$m_{t+1} = \beta_1 m_t + (1 - \beta_1) \nabla L(w_t)$$
$$v_{t+1} = \beta_2 v_t + (1 - \beta_2) (\nabla L(w_t))^2$$
Hiệu chỉnh độ chệch:
$$\hat{m}_{t+1} = \frac{m_{t+1}}{1 - \beta_1^{t+1}}, \quad \hat{v}_{t+1} = \frac{v_{t+1}}{1 - \beta_2^{t+1}}$$
Cập nhật tọa độ:
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{v}_{t+1}} + \epsilon} \odot \hat{m}_{t+1}$$
với các tham số chuẩn: $\beta_1 = 0.9, \beta_2 = 0.999, \epsilon = 10^{-8}$.

In [ ]:
class Adam(BaseOptimizer):
    def __init__(
        self,
        loss_fn: Callable,
        gradient_fn: Callable,
        lr=0.35,
        beta1=0.9,
        beta2=0.999,
        eps=1e-8,
    ):
        super().__init__(loss_fn, gradient_fn)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = None
        self.v = None
        self.t = 0

    def run(self, w_start, num_epochs=140):
        """
        Cập nhật trọng số w theo thuật toán Adam có bias correction.
        """
        if self.m is None:
            self.m = np.zeros_like(w_start)
            self.v = np.zeros_like(w_start)

        # TODO: YOUR CODE HERE
        pass


#### 📊 Trực quan hóa hoạt động của Adam qua 4 kịch bản xuất phát


In [ ]:
# Trực quan hóa độc lập cho Adam trên 2D Contour và Đường cong Loss
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. 2D Contour View
ax_contour = axes[0]
ax_contour.contourf(W1, W2, Z, levels=35, cmap="viridis", alpha=0.9)
c_lines = ax_contour.contour(W1, W2, Z, levels=14, colors="black", alpha=0.3, linewidths=0.6)
ax_contour.clabel(c_lines, inline=True, fontsize=8)

# 2. Loss curve
ax_loss = axes[1]

sc_colors = {
    "Vùng yên ngựa phẳng (Saddle Flat)": "#4ade80",
    "Sườn dốc gần đỉnh núi (Steep Slope)": "#38bdf8",
    "Hẻm núi mấp mô (Rugged Ravine)": "#e879f9",
    "Góc đối diện xa (Far Basin Area)": "#fbbf24"
}

for sc_name, pt in start_scenarios.items():
    opt_instance = Adam(
        loss_fn=loss_function,
        gradient_fn=compute_gradients,
        lr=0.35,
        beta1=0.9,
        beta2=0.999,
    )
    traj, losses = opt_instance.run( pt, num_epochs=140)
    col = sc_colors[sc_name]
    
    # Vẽ trên Contour
    ax_contour.plot(traj[:, 0], traj[:, 1], color=col, linewidth=2.2, label=f"Path: {sc_name}")
    ax_contour.scatter(traj[0, 0], traj[0, 1], color=col, edgecolors="black", s=90, marker="D", zorder=5)
    ax_contour.scatter(traj[-1, 0], traj[-1, 1], color=col, edgecolors="red", s=110, marker="*", zorder=6)
    
    # Vẽ trên Loss curve
    ax_loss.plot(losses, color=col, linewidth=2.0, label=f"Loss: {sc_name}")

ax_contour.scatter([0.5], [-3.5], color="red", edgecolors="white", s=160, marker="*", label="Global Min (0.5, -3.5)", zorder=7)
ax_contour.set_title("Adam - 2D Contour Trajectory View", fontsize=13, fontweight="bold")
ax_contour.set_xlabel("$", fontsize=11)
ax_contour.set_ylabel("$", fontsize=11)
ax_contour.set_xlim(-5, 5)
ax_contour.set_ylim(-5, 5)
ax_contour.legend(loc="lower left", fontsize=8.5, framealpha=0.9)

ax_loss.set_title("Adam - Loss Decay Curve Across 4 Scenarios", fontsize=13, fontweight="bold")
ax_loss.set_xlabel("Epoch", fontsize=11)
ax_loss.set_ylabel("Loss Value", fontsize=11)
ax_loss.grid(True, linestyle="--", alpha=0.5)
ax_loss.legend(fontsize=8.5, framealpha=0.9)

plt.tight_layout()
plt.show()


---
## PHẦN 3: TIẾN HÀNH THỰC NGHIỆM TỐI ƯU HÓA TRÊN ĐA DẠNG ĐIỂM XUẤT PHÁT

Hàm tiện ích `create_optimizers()` khởi tạo lại các instance mới của từng thuật toán để đảm bảo trạng thái bộ nhớ (velocity, gradient accumulator) hoàn toàn độc lập giữa các kịch bản.

In [ ]:
def create_optimizers() -> dict[str, BaseOptimizer]: 
    return {
        "Gradient Descent": GradientDescent(
            loss_fn=loss_function, gradient_fn=compute_gradients, lr=0.04
        ),
        "Momentum": Momentum(
            loss_fn=loss_function, gradient_fn=compute_gradients, lr=0.035, gamma=0.88
        ),
        "AdaGrad": AdaGrad(
            loss_fn=loss_function, gradient_fn=compute_gradients, lr=0.25
        ),
        "RMSProp": RMSProp(
            loss_fn=loss_function, gradient_fn=compute_gradients, lr=0.035, beta=0.9
        ),
        "Adam": Adam(
            loss_fn=loss_function,
            gradient_fn=compute_gradients,
            lr=0.35,
            beta1=0.9,
            beta2=0.999,
        ),
    }

# Chay thuc nghiem tren toan bo 4 kich ban xuat phat
all_scenario_results = {}
epochs = 140

for sc_name, w_start in start_scenarios.items():
    print(f"=== Chạy kịch bản: {sc_name} (w_0 = {w_start}) ===")
    opts = create_optimizers()
    sc_res = {}
    for opt_name, opt in opts.items():
        traj, losses = opt.run(w_start, num_epochs=epochs)
        sc_res[opt_name] = {"traj": traj, "loss": losses}
        print(f"  [{opt_name:<16}] Điểm kết thúc: ({traj[-1, 0]:.3f}, {traj[-1, 1]:.3f}) | Loss cuối: {losses[-1]:.4f}")
    all_scenario_results[sc_name] = sc_res
    print()


---
## PHẦN 4: TRỰC QUAN HÓA SO SÁNH QUỸ ĐẠO & ĐƯỜNG CONG HỘI TỤ

### 1. Trực quan hóa chi tiết cả 4 kịch bản xuất phát trên 2D Contour (Multi-panel Grid)

In [ ]:
opt_colors = {
    'Gradient Descent': '#f87171',
    'Momentum': '#fb923c',
    'AdaGrad': '#facc15',
    'RMSProp': '#38bdf8',
    'Adam': '#c084fc'
}

opt_markers = {
    'Gradient Descent': 'o',
    'Momentum': '^',
    'AdaGrad': 'v',
    'RMSProp': 'x',
    'Adam': '*'
}

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flatten()

for idx, (sc_name, w_start) in enumerate(start_scenarios.items()):
    ax = axes[idx]
    sc_res = all_scenario_results[sc_name]
    
    # Vẽ contour nền
    ax.contourf(W1, W2, Z, levels=35, cmap='viridis', alpha=0.9)
    c_lines = ax.contour(W1, W2, Z, levels=14, colors='black', alpha=0.3, linewidths=0.6)
    ax.clabel(c_lines, inline=True, fontsize=8)
    
    # Vẽ điểm xuất phát và cực tiểu toàn cục
    ax.scatter(w_start[0], w_start[1], color='lime', edgecolors='black', s=130, marker='D', zorder=10, label='Start Point')
    ax.scatter([0.5], [-3.5], color='red', edgecolors='white', s=140, marker='*', zorder=10, label='Global Min')
    
    # Vẽ quỹ đạo của 5 thuật toán
    for opt_name, res in sc_res.items():
        traj = res['traj']
        ax.plot(traj[:, 0], traj[:, 1], label=opt_name, color=opt_colors[opt_name], linewidth=2.2, alpha=0.85)
        ax.scatter(traj[-1, 0], traj[-1, 1], color=opt_colors[opt_name], edgecolors='black', s=80, marker=opt_markers[opt_name], zorder=9)
        
    ax.set_title(f'Kịch bản {idx+1}: {sc_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('$w_1$')
    ax.set_ylabel('$w_2$')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.legend(loc='lower left', framealpha=0.9, fontsize=9)

plt.tight_layout()
plt.show()

### 2. So sánh đường cong suy giảm Loss (Loss vs Epochs) qua 4 kịch bản xuất phát

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for idx, (sc_name, _) in enumerate(start_scenarios.items()):
    ax = axes[idx]
    sc_res = all_scenario_results[sc_name]
    
    for opt_name, res in sc_res.items():
        losses = res['loss']
        ax.plot(losses, label=opt_name, color=opt_colors[opt_name], linewidth=2.2)
        
    ax.set_title(f'Đường cong Loss: {sc_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss Value')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(framealpha=0.9, fontsize=9)

plt.tight_layout()
plt.show()

### 3. Đồ thị 3D Surface biểu diễn quỹ đạo các thuật toán từ vị trí thử thách nhất

In [ ]:
selected_sc = 'Sườn dốc gần đỉnh núi (Steep Slope)'
sc_res = all_scenario_results[selected_sc]
pt_start = start_scenarios[selected_sc]

fig = plt.figure(figsize=(13, 9))
ax = fig.add_subplot(1, 1, 1, projection='3d')

# Vẽ bề mặt 3D
ax.plot_surface(W1, W2, Z, cmap='viridis', edgecolor='none', alpha=0.6)

# Vẽ quỹ đạo của từng thuật toán trên không gian 3 chiều
for name, res in sc_res.items():
    traj = res['traj']
    losses = res['loss']
    ax.plot(traj[:, 0], traj[:, 1], losses, label=name, color=opt_colors[name], linewidth=2.5, zorder=10)
    ax.scatter(traj[-1, 0], traj[-1, 1], losses[-1], color=opt_colors[name], s=60, marker=opt_markers[name])

# Điểm xuất phát chung
z_start = loss_function(pt_start[0], pt_start[1])
ax.scatter(pt_start[0], pt_start[1], z_start, color='lime', edgecolors='black', s=120, marker='D', label='Start')

ax.set_title(f'Mô phỏng quỹ đạo 3D: {selected_sc}', fontsize=13, fontweight='bold')
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_zlabel('Loss')
ax.view_init(elev=52, azim=-118)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

---
## PHẦN 5: BẢNG SO SÁNH ĐỊNH LƯỢNG TỔNG HỢP QUA TẤT CẢ CÁC ĐIỂM XUẤT PHÁT

Bảng tổng hợp đánh giá khả năng hội tụ của các thuật toán theo từng vị trí khởi tạo:

In [ ]:
for sc_name, sc_res in all_scenario_results.items():
    print("=" * 90)
    print(f"Kịch bản: {sc_name.upper()}")
    print("-" * 90)
    print(f"{'Thuật toán (Optimizer)':<20} | {'Tọa độ cuối (w1, w2)':<24} | {'Loss cuối':<12} | {'Đánh giá hội tụ'}")
    print("-" * 90)
    
    for opt_name, res in sc_res.items():
        traj = res['traj']
        losses = res['loss']
        w_end = f"({traj[-1, 0]:.3f}, {traj[-1, 1]:.3f})"
        final_loss = f"{losses[-1]:.4f}"
        
        # Kiểm tra khoảng cách tới Global Minimum (0.5, -3.5)
        dist_to_global = np.linalg.norm(traj[-1] - np.array([0.5, -3.5]))
        if dist_to_global < 0.8:
            status = "🎯 Chạm Cực Tiểu Toàn Cục"
        elif losses[-1] < 2.0:
            status = "⚠️ Kẹt Cực Tiểu Địa Phương"
        else:
            status = "❌ Hội tụ chậm / Chưa tới đáy"
            
        print(f"{opt_name:<20} | {w_end:<24} | {final_loss:<12} | {status}")
    print("=" * 90 + "\n")

---
## PHẦN 6: CÂU HỎI THẢO LUẬN & BÁO CÁO (DÀNH CHO SINH VIÊN)

Sinh viên quan sát kết quả từ **4 kịch bản khởi tạo** và trả lời các câu hỏi sau:

1. **Độ nhạy của thuật toán đối với điểm khởi tạo (Sensitivity to Initialization):**
   - So sánh kết quả của Gradient Descent khi xuất phát từ Vùng yên ngựa phẳng so với Sườn dốc gần đỉnh núi. Vì sao điểm khởi tạo lại ảnh hưởng quyết định đến cực tiểu mà GD tìm được?

2. **Thoát bẫy địa phương và kiểm soát quán tính:**
   - Ở kịch bản *Hẻm núi mấp mô* hoặc *Sườn dốc*, Momentum và Adam có vượt qua được các bẫy cực tiểu cạn để tiến gần đến Cực tiểu toàn cục $(0.5, -3.5)$ không? Giải thích vai trò của quán tính vận tốc $v_t$.
   - Trường hợp nào quán tính lớn có thể gây tác dụng phụ (overshoot / dao động mạnh)?

3. **Tính thích ứng của AdaGrad vs RMSProp vs Adam:**
   - Trong cả 4 kịch bản, thuật toán nào thể hiện tính thích nghi mạnh mẽ nhất trên các dạng địa hình đa dạng (vùng phẳng, sườn dốc cao, lòng hẹp)? Hãy rút ra kết luận lý do Adam được coi là bộ tối ưu hóa tiêu chuẩn vàng trong Deep Learning.